<a href="https://colab.research.google.com/github/sandeshdevre25/data-engineering/blob/main/DE_prac_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import pandas as pd
import requests

# 1. Extract data from REST API
def extract_api_data(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        return pd.json_normalize(data)
    except requests.exceptions.RequestException as e:
        print("API Error:", e)
        return pd.DataFrame()


# 2. Extract data from CSV file
def extract_csv_data(file_path):
    try:
        return pd.read_csv(file_path)
    except FileNotFoundError:
        print("CSV file not found.")
        return pd.DataFrame()


if __name__ == "__main__":

    # Fetch user data from API
    api_url = "https://jsonplaceholder.typicode.com/users"
    api_df = extract_api_data(api_url)

    if not api_df.empty:
        api_df = api_df[
            ["id", "name", "email", "company.name"]
        ]
        api_df.rename(
            columns={"company.name": "company"},
            inplace=True
        )

        # Clean missing and duplicate records
        api_df.dropna(subset=["name", "email"], inplace=True)
        api_df.drop_duplicates(subset=["id"], inplace=True)

    # 3. Create sample location CSV file
    locations = pd.DataFrame({
        "id": list(range(1, 11)),
        "city": [
            "New York", "London", "Paris", "Tokyo", "Berlin",
            "Delhi", "Sydney", "Moscow", "Cairo", "Beijing"
        ],
        "country": [
            "USA", "UK", "France", "Japan", "Germany",
            "India", "Australia", "Russia", "Egypt", "China"
        ]
    })

    locations.to_csv("locations.csv", index=False)

    # Read CSV data
    csv_df = extract_csv_data("locations.csv")

    # 4. Merge API and CSV data
    if not api_df.empty and not csv_df.empty:
        merged_df = pd.merge(
            api_df, csv_df, on="id", how="inner"
        )

        print("\n--- Merged User Data ---")
        print(merged_df)

        # Save final cleaned data
        merged_df.to_csv(
            "cleaned_warehouse_profiles.csv",
            index=False
        )

        print("\nData saved successfully!")
    else:
        print("Unable to merge data.")


--- Merged User Data ---
   id                      name                      email  \
0   1             Leanne Graham          Sincere@april.biz   
1   2              Ervin Howell          Shanna@melissa.tv   
2   3          Clementine Bauch         Nathan@yesenia.net   
3   4          Patricia Lebsack  Julianne.OConner@kory.org   
4   5          Chelsey Dietrich   Lucio_Hettinger@annie.ca   
5   6      Mrs. Dennis Schulist    Karley_Dach@jasper.info   
6   7           Kurtis Weissnat     Telly.Hoeger@billy.biz   
7   8  Nicholas Runolfsdottir V       Sherwood@rosamond.me   
8   9           Glenna Reichert    Chaim_McDermott@dana.io   
9  10        Clementina DuBuque     Rey.Padberg@karina.biz   

              company      city    country  
0     Romaguera-Crona  New York        USA  
1        Deckow-Crist    London         UK  
2  Romaguera-Jacobson     Paris     France  
3       Robel-Corkery     Tokyo      Japan  
4         Keebler LLC    Berlin    Germany  
5   Considine-Lockman